In [1]:
from src.connection.clinets import weaviate_client
from typing import List, Any
import weaviate
import pandas as pd
from langchain_weaviate import WeaviateVectorStore
from sentence_transformers import SentenceTransformer

/home/codespace/.local/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class SentenceTransformersEmbeddings:
    def __init__(self, model_name: str):
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        # returns a list of embeddings for documents
        return self.model.encode(texts).tolist()

    def embed_query(self, text: str) -> List[float]:
        # returns a single embedding for a query
        return self.model.encode([text])[0].tolist()

In [3]:
embedding_model = SentenceTransformersEmbeddings('sentence-transformers/all-mpnet-base-v2')

Loading weights: 100%|██████████| 199/199 [00:07<00:00, 25.95it/s, Materializing param=pooler.dense.weight]                         
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [4]:
laws = pd.read_csv('dataset/Celex_act_raw_text.csv')

In [22]:
def get__embeddings(enhanced_query):
    query_embedding = embedding_model.encode([enhanced_query]).tolist()[0]

    return query_embedding

In [5]:
vectorstore = WeaviateVectorStore(
    client = weaviate_client,
    index_name = "Euro_Laws",
    text_key="text",
    embedding = embedding_model
)

In [ ]:
def search_docs(query):
    celex_ids = []

    retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
    docs = retriever.invoke(query)

    for doc in docs:
        celex_ids.append(doc.metadata['celex'])

    celex_ids = list[Any](set(celex_ids))    

    for celex_id in celex_ids:
        doc = laws[laws['CELEX'] == celex_id]['act_raw_text'].iloc[0]    





>> 11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking THE COUNCIL OF THE EUROPEAN UNION, Having regard to the Treaty on European Union, and in particular Article 31(e) and Article 34(2)(b) thereof, Having regard to the proposal from the Commission (1), Having regard to the opinion of the European Parliament (2), Whereas: (1) Illicit drug trafficking poses a threat to health, safety and the quality of life of citizens of the European Union, and to the legal economy, stability and security of the Member States. (2) The need for legislative action to tackle illicit drug trafficking has been recognised in particular in the Action Plan of the Council and the Commission on how best to implement the provisions of the Amsterdam Treaty on an area of freedom, security and justice (3), adopted by the

In [ ]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})
docs = retriever.invoke("Drug dealing Sentences")

retrived = []

for i , doc in enumerate (docs):
    retrived.append([f"Doc{i}:"  doc.metadata['status','treaty','act_type','celex']])


[Document(metadata={'subject_matter': 'sources and branches of the law;  European Union law;  justice;  criminal law', 'status': 'In Force', 'legal_basis': '12002M031; 12002M034', 'additional_info': 'CNS 2001/0114', 'chunk_number': 1, 'eurovoc': 'penal code; Community law - national law; criminal procedure; penalty; drug traffic', 'treaty': 'TEU (1992)', 'act_type': 'Decision_FRAMW', 'cites': 'joint_action/1997/396; 31999Y0123%2801%29; joint_action/1998/733', 'celex': '32004F0757', 'authors': 'European Council', 'act_name': 'Council Framework Decision 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the field of illicit drug trafficking', 'document_length': 13663, 'total_chunks': 6}, page_content="11.11.2004 EN Official Journal of the European Union L 335/8 COUNCIL FRAMEWORK DECISION 2004/757/JHA of 25 october 2004 laying down minimum provisions on the constituent elements of criminal acts and penalties in the 

In [6]:
retrived = []

for doc in docs:
    retrived.append({doc.metadata['status','treaty']})

KeyError: ('status', 'treaty')

In [45]:
docs[0].metadata['celex']

'32004F0757'

In [ ]:
def get_celex_ids(docs):
    celex_ids = []
    for i in docs:
        celex_ids.append(docs[i].metadata['celex'])

    celex_ids = list(set(celex_ids))

    return celex_ids    

In [1]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI."),
    ("human", "Answer this question: {question}")
])

formatted = prompt.invoke({"question": "What is AI?"})
print(formatted)

messages=[SystemMessage(content='You are a helpful AI.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Answer this question: What is AI?', additional_kwargs={}, response_metadata={})]


In [9]:
laws[laws['CELEX'] == '31997R2046']['act_raw_text'].iloc[0]   

"Avis juridique important|31997R2046Council Regulation (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addiction Official Journal L 287 , 21/10/1997 P. 0001 - 0005COUNCIL REGULATION (EC) No 2046/97 of 13 October 1997 on north-south cooperation in the campaign against drugs and drug addictionTHE COUNCIL OF THE EUROPEAN UNION,Having regard to the Treaty establishing the European Community, and in particular Article 130w thereof,Having regard to the proposal from the Commission (1),Acting in accordance with the procedure laid down in Article 189c of the Treaty (2),Whereas the impact on the structures of a developing society of an economy based on the production of drugs, or which derives a substantial revenue from them, undermines a country's smooth integration into the world economy;Whereas the breakdown of social structures in developing countries due to drug consumption and the related industry is detrimental to sustainable social de